In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch09-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "tree-ensembles.py"

# 同一批分子上的單樹、森林與提升樹

Colab 請選 2026.07，第一次載入實驗資料需要連網。下格寫入暫存程式，最後一格執行並顯示圖。做小練習時，把程式接在分析格下方，重新執行兩格。

致突變性資料取自 Lou 等人（2023）的 [Maom Lab 鏡像](https://huggingface.co/datasets/maomlab/MutagenLou2023)（CC BY 4.0）。沿用鏡像測試檔，清理結構後從開發資料分出驗證集。

In [ ]:
%%writefile "{script}"
# MutagenLou2023 (Maom Lab mirror, CC BY 4.0; Lou et al., 2023).
# Original rows/labels preserved in the cached files. Only structure enters X.
from pathlib import Path
from urllib.request import urlopen
import csv
import hashlib
import io
import os
import tempfile
import numpy as np
from rdkit import Chem
from sklearn.model_selection import train_test_split

def mutagen_splits():
    revision = "a9780a60774eb522b78bd92b83c30d8deae00ff5"
    source = f"https://huggingface.co/datasets/maomlab/MutagenLou2023/resolve/{revision}/train_test"
    base = Path(os.environ.get("DEEPCHEM_DATA_DIR", tempfile.gettempdir())) / "mutagenlou-a9780a6"
    base.mkdir(parents=True, exist_ok=True)
    expected = {"train": "ad0d4e2850afdc07c130c3679cc8ae2ad9f7c80999158325d8de986c39f950e1",
                "test": "08786199fbbdb6db614d5f65a5abdfcd26d882f0cc844bbeb4ccfa241127b394"}
    groups = {}
    for name, digest in expected.items():
        file = base / (name + ".csv")
        raw = file.read_bytes() if file.exists() else urlopen(f"{source}/{name}.csv", timeout=90).read()
        if hashlib.sha256(raw).hexdigest() != digest:
            raise ValueError(f"MutagenLou2023 checksum mismatch: {name}")
        if not file.exists():
            file.write_bytes(raw)
        labels = {}
        for row in csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))):
            mol = Chem.MolFromSmiles(row["SMILES"])
            if mol is not None:
                y = int(row["Y"])
                assert y in (0, 1)
                labels.setdefault(Chem.MolToSmiles(mol), set()).add(y)
        groups[name] = labels
    # Preserve mirrored test membership: no test structure may enter development.
    dev = [(s, next(iter(v))) for s,v in groups["train"].items() if len(v)==1 and s not in groups["test"]]
    test = [(s, next(iter(v))) for s,v in groups["test"].items() if len(v)==1]
    fit, valid = train_test_split(dev, test_size=0.15, random_state=42, stratify=[y for _,y in dev])
    return [(np.array([s for s,_ in rows]), np.array([y for _,y in rows])) for rows in (fit,valid,test)]

from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

SEED = 42
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

class MorganBits1024(dc.feat.MolecularFeaturizer):
    def _featurize(self, mol, **kwargs):
        return generator.GetFingerprintAsNumPy(mol)

options = dict(featurizer=MorganBits1024(), splitter="scaffold", transformers=[],
               data_dir=str(cache), save_dir=str(cache), reload=False)


def make_dataset(smiles, y, featurize):
    X = np.stack([featurize(Chem.MolFromSmiles(s)) for s in smiles])
    return dc.data.NumpyDataset(X=X, y=y, ids=smiles)

from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.metrics import mean_absolute_error, roc_auc_score

# ESOL, 1,128 compounds, log10(mol/L), DOI 10.1021/ci034243x
tasks, (train_r, valid_r, test_r), _ = dc.molnet.load_delaney(**options)
assert tasks == ["measured log solubility in mols per litre"]
train_c, valid_c, test_c = [make_dataset(s, y, generator.GetFingerprintAsNumPy) for s, y in mutagen_splits()]
assert all(np.all(d.w > 0) for d in (train_r, valid_r, test_r))
reg = {"Tree": DecisionTreeRegressor(random_state=SEED),
       "Forest": RandomForestRegressor(n_estimators=100, max_features=0.33, random_state=SEED, n_jobs=1),
       "Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=SEED)}
cls = {"Tree": DecisionTreeClassifier(random_state=SEED),
       "Forest": RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=SEED, n_jobs=1),
       "Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=SEED)}

def mae(model, data):
    return mean_absolute_error(data.y.ravel(), model.predict(data.X))

def auc(model, data):
    p = model.predict_proba(data.X)[:, list(model.classes_).index(1)]
    return roc_auc_score(data.y.ravel(), p)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
for ax, models, train, valid, test, metric, choose, name in [
    (axes[0], reg, train_r, valid_r, test_r, mae, min, "ESOL MAE"),
    (axes[1], cls, train_c, valid_c, test_c, auc, max, "MutagenLou2023 ROC-AUC"),
]:
    scores = {}
    for label, model in models.items():
        model.fit(train.X, train.y.ravel())
        scores[label] = metric(model, valid)
        print(name, label, "train / valid", f"{metric(model, train):.3f}", f"{scores[label]:.3f}")
    winner = choose(scores, key=scores.get)
    print(name, winner, "test", f"{metric(models[winner], test):.3f}")
    ax.bar(scores.keys(), scores.values(), color=["#969a9b", "#33748b", "#b76138"])
    ax.set(title=name, ylabel="Validation MAE (log mol/L)" if metric is mae else "Validation ROC-AUC")
fig.savefig(figure_dir / "09-07-tree-ensembles.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-07-tree-ensembles.png")))